In [6]:
# A simple turn-based sci fi game



# game setup for new game

# hero selection
print("Welcome to the RPG!")
print("Please select your hero.")

def hero_selection():
    hero_name = input("Hero name: ")
    hero_type = input("Please select your hero type: 1) Sorcerer, 2) Thug, 3) Slayer, 4) Sneak, 5) Genius")
    hero_gift = input("Please select the gift you want to give your hero: 1) Speech, 2) Speed, 3) Physical Strength, 4) Sight, 5) Love")
    magic_system = input("Please select your magic system: 1) Fire, 2) Water, 3) Earth, 4) Air, 5) Lightning, 6) Darkness, 7) Nano")
    hero_home = input("Please select your hero's home: 1) Forest, 2) Desert, 3) Mountain, 4) Sea, 5) Sky")
    return hero_name, hero_type, hero_gift, magic_system, hero_home
hero_name, hero_type, hero_gift, magic_system, hero_home = hero_selection()
print(f"Your hero is {hero_name}, a {hero_type} with a gift for {hero_gift} and a magic system of {magic_system} from {hero_home}.")

print("You are ready to begin the epic journey!")

#introduction to the journey

import pygame
import math
import random

pygame.init()
pygame.mixer.init(frequency=44100, size=-16, channels=1, buffer=512)
W, H = 800, 450
screen = pygame.display.set_mode((W, H))
pygame.display.set_caption("2157")
clock = pygame.time.Clock()

# ── Palette ──
SKY_TOP  = (8,   4,  20)
SKY_BOT  = (25, 10,  45)
PINK     = (220, 40, 120)
CYAN     = (0,  200, 220)
PURPLE   = (120, 40, 200)
YELLOW   = (220, 180,  0)
DIM_CYAN = (0,   80,  90)
DIM_PINK = (80,  15,  45)
GRID_COL = (0,   60,  80)

# ── Crash sound (synthesised — no file needed) ──
def make_crash_sound():
    rate     = 44100
    duration = 2.2          # seconds
    n        = int(rate * duration)
    buf      = bytearray(n)

    for i in range(n):
        t   = i / rate
        env = max(0.0, 1.0 - t / duration) ** 0.4   # decay envelope

        # white-noise burst
        noise = (random.randint(0, 255) - 128) * env

        # low-freq thud at the start
        thud_freq = 55
        thud      = math.sin(2 * math.pi * thud_freq * t) * 127 * max(0, 1 - t * 4)

        # high screech that drops in pitch
        screech_freq = max(40, 1800 - t * 1400)
        screech      = math.sin(2 * math.pi * screech_freq * t) * 60 * env * min(1, t * 8)

        sample = int(noise * 0.55 + thud + screech)
        buf[i] = max(0, min(255, sample + 128))

    sound = pygame.mixer.Sound(buffer=bytes(buf))
    return sound

crash_sound = make_crash_sound()

# ── Scroll text ──
RAW_TEXT = (
    "THE WORLD — 2157 A.D.\n"
    "\n"
    "The year is 2157. Earth is unrecognizable\n"
    "yet strangely familiar.\n"
    "\n"
    "Three generations have passed since the\n"
    "Singularity Accord — the moment humanity's\n"
    "greatest minds, governments, and spiritual\n"
    "leaders convened to answer one question\n"
    "that could no longer be avoided:\n"
    "\n"
    "What do we do now that we can do anything?\n"
    "\n"
    "The answer fractured civilization into\n"
    "three distinct paths, known collectively\n"
    "as The Pitchfork — a term coined\n"
    "sarcastically by a late-night comedian\n"
    "in 2089 that somehow stuck, eventually\n"
    "becoming the official name in global\n"
    "census records.\n"
    "\n"
    "The three tines of the Pitchfork have\n"
    "since calcified into cultures, economies,\n"
    "ideologies, and wars.\n"
    "\n"
    "The player exists at the bleeding edge\n"
    "of all three.\n"
    "\n\n\n\n\n\n"          # padding so last line clears the screen
)

pygame.font.init()
FONT_TITLE  = pygame.font.SysFont("consolas", 15, bold=True)
FONT_BODY   = pygame.font.SysFont("consolas", 13)
FONT_HUD    = pygame.font.SysFont("consolas", 11)
FONT_HERO   = pygame.font.SysFont("consolas", 38, bold=True)
FONT_HERO_S = pygame.font.SysFont("consolas", 20, bold=True)

def make_text_surfaces(raw):
    surfaces = []
    for line in raw.split("\n"):
        stripped = line.strip()
        if stripped == "":
            surfaces.append(None)
            continue
        if stripped.startswith("THE WORLD"):
            surf = FONT_TITLE.render(line, True, CYAN)
        elif stripped.startswith("What do we"):
            surf = FONT_BODY.render(line, True, YELLOW)
        else:
            surf = FONT_BODY.render(line, True, (160, 200, 220))
        surfaces.append(surf)
    return surfaces

text_surfaces  = make_text_surfaces(RAW_TEXT)
LINE_H         = 22
SCROLL_SPEED   = 0.55
text_y         = float(H)
TEXT_X         = 30
TEXT_W         = W - 60
PANEL_ALPHA    = 170

# How far text_y must scroll before all lines have left the top
total_text_h   = len(text_surfaces) * LINE_H
# text is "done" when text_y + total_text_h < 0  →  text_y < -total_text_h
text_done      = False

# ── Stars ──
random.seed(42)
stars = [
    (random.randint(0,W), random.randint(0,H//2),
     random.random(), random.choice([CYAN,PINK,(180,180,220)]))
    for _ in range(120)
]

# ── Moon ──
moon = {"x":620,"y":80,"r":38}

# ── Buildings ──
def make_buildings():
    blds   = []
    ground = int(H*0.68)
    specs_back  = [(30,160),(90,210),(160,180),(230,230),(300,170),
                   (370,250),(440,190),(510,220),(580,160),(650,200),(710,175),(760,140)]
    specs_mid   = [(0,110),(60,130),(130,100),(200,120),(280,140),
                   (350,110),(420,130),(490,120),(560,100),(630,130),(700,115),(755,95)]
    specs_front = [(0,55),(70,65),(150,50),(230,70),(310,58),
                   (390,62),(460,55),(535,68),(610,52),(675,60),(735,55),(775,48)]
    for layer,(specs,wrange,wc) in enumerate([
        (specs_back, (44,68), PURPLE),
        (specs_mid,  (52,72), CYAN),
        (specs_front,(55,78), YELLOW),
    ]):
        cols = [(18,8,40),(12,6,28),(6,3,16)][layer]
        for x,h in specs:
            w = random.randint(*wrange)
            blds.append({
                "x":x,"y":ground-h,"w":w,"h":h,
                "color":cols,"layer":layer,"win_color":wc,
                "windows":[(random.randint(4,w-10),random.randint(6,h-10))
                           for _ in range(random.randint(2,12))
                           if random.random()>0.3]
            })
    blds.sort(key=lambda b:b["layer"])
    return blds, ground

buildings, GROUND_Y = make_buildings()
win_states = {i:[random.random()>0.15 for _ in b["windows"]]
              for i,b in enumerate(buildings)}

# ── Rain ──
drops = [{"x":random.uniform(0,W),"y":random.uniform(0,H),
           "speed":random.uniform(4,9),"length":random.randint(6,16),
           "color":random.choice([DIM_CYAN,DIM_PINK,(40,30,70)])}
          for _ in range(140)]

# ── Flyers ──
flyers = [{"x":random.uniform(0,W),"y":random.uniform(H*0.1,H*0.55),
            "speed":random.uniform(0.6,2.2)*random.choice([-1,1]),
            "color":random.choice([CYAN,YELLOW,PINK,(180,180,255)]),
            "size":random.randint(2,4),"blink":random.randint(20,60)}
           for _ in range(8)]

# ── Helpers ──
def draw_sky():
    for y in range(H):
        t = y/H
        r = int(SKY_TOP[0]+(SKY_BOT[0]-SKY_TOP[0])*t)
        g = int(SKY_TOP[1]+(SKY_BOT[1]-SKY_TOP[1])*t)
        b = int(SKY_TOP[2]+(SKY_BOT[2]-SKY_TOP[2])*t)
        pygame.draw.line(screen,(r,g,b),(0,y),(W,y))

def draw_stars(tick):
    for sx,sy,phase,sc in stars:
        bright = 0.5+0.5*math.sin(tick*0.05+phase*8)
        pygame.draw.circle(screen,tuple(int(c*bright) for c in sc),(sx,sy),1)

def draw_moon():
    for gr in range(22,0,-4):
        gs = pygame.Surface((gr*2,gr*2),pygame.SRCALPHA)
        pygame.draw.circle(gs,PINK+(int(40*gr/22),),(gr,gr),gr)
        screen.blit(gs,(moon["x"]-gr,moon["y"]-gr))
    pygame.draw.circle(screen,(35,12,80),(moon["x"],moon["y"]),moon["r"])
    pygame.draw.circle(screen,(55,18,110),(moon["x"],moon["y"]),moon["r"],2)
    for cx,cy,cr in [(608,72,7),(632,88,5),(618,95,4)]:
        pygame.draw.circle(screen,(28,9,65),(cx,cy),cr)

def draw_horizon(ground_y):
    for i in range(30):
        t=i/30; a=int(60*(1-t))
        col=(int(PINK[0]*(1-t)+PURPLE[0]*t),
             int(PINK[1]*(1-t)+PURPLE[1]*t),
             int(PINK[2]*(1-t)+PURPLE[2]*t))
        s=pygame.Surface((W,1),pygame.SRCALPHA); s.fill(col+(a,))
        screen.blit(s,(0,ground_y-i))

def draw_buildings(tick):
    for i,b in enumerate(buildings):
        pygame.draw.rect(screen,b["color"],(b["x"],b["y"],b["w"],b["h"]))
        edge=CYAN if b["layer"]==0 else(PINK if b["layer"]==1 else YELLOW)
        pygame.draw.line(screen,edge,(b["x"],b["y"]),(b["x"],b["y"]+b["h"]),1)
        for j,(wx,wy) in enumerate(b["windows"]):
            on=win_states[i][j]
            if tick%random.randint(90,300)==(i+j)%60:
                win_states[i][j]=not on; on=win_states[i][j]
            pygame.draw.rect(screen,b["win_color"] if on else(8,4,18),
                             (b["x"]+wx,b["y"]+wy,5,4))
        if b["layer"] in(0,1) and b["w"]>55:
            ax=b["x"]+b["w"]//2
            pygame.draw.line(screen,(80,80,100),(ax,b["y"]),(ax,b["y"]-14),1)
            pygame.draw.circle(screen,PINK if(tick//25)%2==0 else(30,10,30),(ax,b["y"]-14),2)

def draw_ground(tick):
    pygame.draw.rect(screen,(4,1,12),(0,GROUND_Y,W,H-GROUND_Y))
    vp=W//2
    for gx in range(0,W+1,55):
        pygame.draw.line(screen,GRID_COL,(vp,GROUND_Y),(gx,H),1)
    spacing=32; offset=(tick*2)%spacing
    for gy in range(0,H-GROUND_Y+spacing,spacing):
        t=gy/(H-GROUND_Y+spacing); br=int(80*t)
        col=(0,min(br,80),min(br+20,100))
        y=GROUND_Y+gy-offset
        if GROUND_Y<=y<=H: pygame.draw.line(screen,col,(0,y),(W,y),1)

def draw_rain():
    rs=pygame.Surface((W,H),pygame.SRCALPHA)
    for d in drops:
        d["y"]+=d["speed"]; d["x"]+=d["speed"]*0.12
        if d["y"]>H: d["y"]=random.uniform(-10,0); d["x"]=random.uniform(0,W)
        pygame.draw.line(rs,d["color"]+(55,),
                         (int(d["x"]),int(d["y"])),
                         (int(d["x"]+d["length"]*0.12),int(d["y"]+d["length"])),1)
    screen.blit(rs,(0,0))

def draw_flyers(tick):
    for f in flyers:
        f["x"]+=f["speed"]
        if f["x"]>W+20: f["x"]=-20
        if f["x"]<-20:  f["x"]=W+20
        if(tick//f["blink"])%2==0:
            pygame.draw.circle(screen,f["color"],(int(f["x"]),int(f["y"])),f["size"])

def draw_scanlines():
    for sy in range(0,H,4):
        pygame.draw.line(screen,(0,0,0),(0,sy),(W,sy),1)

def draw_scroll_text(text_y):
    panel=pygame.Surface((TEXT_W,H),pygame.SRCALPHA)
    panel.fill((4,1,14,PANEL_ALPHA))
    screen.blit(panel,(TEXT_X,0))
    for idx,surf in enumerate(text_surfaces):
        ly=int(text_y+idx*LINE_H)
        if -LINE_H<ly<H and surf is not None:
            fade=255
            if ly>H-60: fade=int(255*(H-ly)/60)
            if ly<60:   fade=int(255*ly/60)
            fade=max(0,min(255,fade))
            glow=surf.copy(); glow.set_alpha(max(0,fade-160))
            screen.blit(glow,(TEXT_X+10,ly+1))
            lx=TEXT_X+(TEXT_W-surf.get_width())//2
            surf.set_alpha(fade)
            screen.blit(surf,(lx,ly))

# ════════════════════════════════════════
# STATE MACHINE
# ════════════════════════════════════════
# "scroll"  → normal scrolling skyline
# "hero"    → big "THE HERO'S JOURNEY BEGINS" text
# "pixelate"→ white pixel static builds up
# "crash"   → full white flash then freeze
# "done"    → quit

state        = "scroll"
hero_alpha   = 0          # fades in
hero_timer   = 0          # frames spent in hero state
HERO_HOLD    = 210        # frames (~3.5 s) to show hero text

pixel_timer  = 0
PIXEL_PHASES = 120        # frames for pixelation to build
pixel_surf   = None       # offscreen for pixelated frame

crash_timer  = 0
CRASH_HOLD   = 90
crash_played = False

tick = 0
running = True

while running:
    clock.tick(60)
    tick += 1

    for e in pygame.event.get():
        if e.type == pygame.QUIT:
            running = False
        if e.type == pygame.KEYDOWN and e.key == pygame.K_ESCAPE:
            running = False

    # ════════════════════════════════════
    # DRAW SKYLINE (always, as backdrop)
    # ════════════════════════════════════
    draw_sky()
    draw_stars(tick)
    draw_moon()
    draw_horizon(GROUND_Y)
    draw_buildings(tick)
    draw_ground(tick)
    draw_rain()
    draw_flyers(tick)
    draw_scanlines()

    # ════════════════════════════════════
    # STATE: scroll
    # ════════════════════════════════════
    if state == "scroll":
        text_y -= SCROLL_SPEED
        draw_scroll_text(text_y)

        # HUD
        hud = FONT_HUD.render("NEON DISTRICT  //  2157", True, (0,120,140))
        hud.set_alpha(100)
        screen.blit(hud,(8,8))

        # Check if all text has scrolled off the top
        if text_y + total_text_h < 0:
            text_done = True
            state     = "hero"
            hero_alpha = 0

    # ════════════════════════════════════
    # STATE: hero  — big title fades in
    # ════════════════════════════════════
    elif state == "hero":
        hero_timer += 1

        # Fade in over first 60 frames, hold, done
        if hero_timer < 60:
            hero_alpha = int(255 * hero_timer / 60)
        else:
            hero_alpha = 255

        # Dark vignette so text pops
        veil = pygame.Surface((W, H), pygame.SRCALPHA)
        veil.fill((0, 0, 0, 160))
        screen.blit(veil, (0, 0))

        # Pulsing glow behind text
        pulse = 0.7 + 0.3 * math.sin(tick * 0.08)

        line1 = "THE HERO'S"
        line2 = "JOURNEY BEGINS"

        for font, text, y_off, col in [
            (FONT_HERO,   line1, -52, CYAN),
            (FONT_HERO,   line2,   8, PINK),
            (FONT_HERO_S, "── PRESS ANY KEY ──", 78, (100,160,180)),
        ]:
            rendered = font.render(text, True, col)
            # glow pass
            glow_col = tuple(int(c * pulse * 0.6) for c in col)
            glow     = font.render(text, True, glow_col)
            cx = W//2 - rendered.get_width()//2
            cy = H//2 + y_off
            for ox,oy in [(-3,0),(3,0),(0,-3),(0,3)]:
                g=glow.copy(); g.set_alpha(int(80*pulse))
                screen.blit(g,(cx+ox,cy+oy))
            rendered.set_alpha(hero_alpha)
            screen.blit(rendered,(cx,cy))

        # Advance after hold, OR on any keypress
        keys = pygame.key.get_pressed()
        key_pressed = any(keys)

        if hero_timer >= HERO_HOLD or (hero_timer > 60 and key_pressed):
            state       = "pixelate"
            pixel_timer = 0
            # Snapshot current frame for pixelation
            pixel_surf  = screen.copy()

    # ════════════════════════════════════
    # STATE: pixelate — blocky static
    # ════════════════════════════════════
    elif state == "pixelate":
        pixel_timer += 1
        progress = pixel_timer / PIXEL_PHASES   # 0 → 1

        if not crash_played and progress > 0.3:
            crash_sound.play()
            crash_played = True

        # Draw the snapshot then blast random white/colour squares over it
        screen.blit(pixel_surf, (0, 0))

        # Block size grows as chaos increases
        block = max(2, int(4 + progress * 28))
        num_blocks = int(progress * (W * H) // (block * block) * 1.4)

        noise_surf = pygame.Surface((W, H), pygame.SRCALPHA)
        for _ in range(num_blocks):
            bx = random.randint(0, W - block)
            by = random.randint(0, H - block)
            # Mostly white, occasional colour
            if random.random() < 0.75:
                col = (255, 255, 255, random.randint(120, 255))
            else:
                col = random.choice([
                    CYAN+(random.randint(80,200),),
                    PINK+(random.randint(80,200),),
                    YELLOW+(random.randint(80,200),),
                    (255,255,255,255),
                ])
            noise_surf.fill(col, (bx, by, block, block))

        screen.blit(noise_surf, (0, 0))

        # Horizontal glitch bars
        if random.random() < 0.5 + progress * 0.5:
            for _ in range(int(progress * 12)):
                gy  = random.randint(0, H)
                gh  = random.randint(2, 8)
                gsh = int(progress * 40)
                glitch = pygame.Surface((W, gh), pygame.SRCALPHA)
                glitch.blit(screen, (-random.randint(0,gsh), -gy))
                col = random.choice([CYAN,PINK,YELLOW,(255,255,255)])+(160,)
                pygame.draw.rect(glitch, col, (0,0,W,gh), 1)
                screen.blit(glitch, (0, gy))

        # Screen-shake offset
        shake = int(progress * 9)
        if shake > 0:
            tmp = screen.copy()
            ox = random.randint(-shake, shake)
            oy = random.randint(-shake, shake)
            screen.fill((0,0,0))
            screen.blit(tmp, (ox, oy))

        if pixel_timer >= PIXEL_PHASES:
            state       = "crash"
            crash_timer = 0

    # ════════════════════════════════════
    # STATE: crash — full white, hold, quit
    # ════════════════════════════════════
    elif state == "crash":
        crash_timer += 1

        # Blinding white flash
        screen.fill((255, 255, 255))

        # After a beat, add a tiny "signal lost" message
        if crash_timer > 20:
            sig = FONT_HUD.render("// SIGNAL LOST //", True, (180,0,0))
            screen.blit(sig, (W//2 - sig.get_width()//2, H//2))

        if crash_timer >= CRASH_HOLD:
            state = "done"

    # ════════════════════════════════════
    elif state == "done":
        running = False

    pygame.display.flip()

# Brief pause so the white screen is visible before window closes
pygame.time.wait(400)
pygame.quit()

# loading saved game


Welcome to the RPG!
Please select your hero.
Your hero is Jethro, a 2 with a gift for 4 and a magic system of 5 from 3.
You are ready to begin the epic journey!
